# Phase 4b — ROSE as the editor (removal-specialized video prior)
Runs ROSE (Wan2.1-Fun-1.3B base, transformer fine-tuned for side-effect removal)
on the orbit with both mask protocols. Requires `rose_inputs/` on Drive
(exported by the Phase-4 notebook). Runtime: GPU; no restarts needed.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone ROSE; pin transformers to the version its diffusers dependency expects
import os
%cd /content
if not os.path.isdir('/content/ROSE'):
    !git clone https://github.com/Kunbyte-AI/ROSE.git
%cd /content/ROSE
!pip -q install -r requirements.txt
!pip -q install "transformers==4.46.2" "tokenizers>=0.20,<0.21"
!pip -q install --force-reinstall "huggingface_hub==0.36.2" 

In [ ]:
# Weights: base model + ROSE transformer (HF stores it flat; ROSE expects
# weights/transformer/, so relocate the two files)
from huggingface_hub import snapshot_download
import os, shutil
if not os.path.isdir('/content/ROSE/models/Wan2.1-Fun-1.3B-InP'):
    snapshot_download('alibaba-pai/Wan2.1-Fun-1.3B-InP',
                      local_dir='/content/ROSE/models/Wan2.1-Fun-1.3B-InP')
if not os.path.exists('/content/ROSE/weights/transformer/config.json'):
    snapshot_download('Kunbyte/ROSE', local_dir='/content/ROSE/weights')
    os.makedirs('/content/ROSE/weights/transformer', exist_ok=True)
    for f in ('config.json', 'diffusion_pytorch_model.safetensors'):
        if os.path.exists(f'/content/ROSE/weights/{f}'):
            shutil.move(f'/content/ROSE/weights/{f}',
                        f'/content/ROSE/weights/transformer/{f}')
print('weights ready')

In [ ]:
import shutil, os
DRIVE = '/content/drive/MyDrive/light-footprint-removal'
os.makedirs('/content/inputs', exist_ok=True)
for f in ('orbit.mp4', 'mask_object.mp4', 'mask_oracle.mp4'):
    shutil.copy(f'{DRIVE}/rose_inputs/{f}', f'/content/inputs/{f}')

In [ ]:
# Inference: 81 frames satisfies ROSE's 16n+1 rule; size matches the dataset
%cd /content/ROSE
for mask, out in [('mask_object', 'rose_out_object'), ('mask_oracle', 'rose_out_oracle')]:
    get_ipython().system(
        f'python inference.py --validation_videos /content/inputs/orbit.mp4 '
        f'--validation_masks /content/inputs/{mask}.mp4 '
        f'--validation_prompts "remove the object" '
        f'--output_dir /content/{out} --video_length 81 --sample_size 480 832')

In [ ]:
# Decode output videos to frame folders; visual check
import glob, os
import imageio.v3 as iio
from PIL import Image
import matplotlib.pyplot as plt

def to_frames(out_dir, tag):
    vid = sorted(glob.glob(f'{out_dir}/**/*.mp4', recursive=True))[0]
    d = f'/content/edited_{tag}/rgb'
    os.makedirs(d, exist_ok=True)
    for i, fr in enumerate(iio.imread(vid, plugin='pyav'), start=1):
        Image.fromarray(fr).resize((832, 480)).save(f'{d}/{i:04d}.png')
    return f'/content/edited_{tag}'

d1 = to_frames('/content/rose_out_object', 'rose_objectmask')
d2 = to_frames('/content/rose_out_oracle', 'rose_oraclemask')
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(Image.open(f'{d1}/rgb/0041.png')); ax[0].set_title('ROSE: object mask')
ax[1].imshow(Image.open(f'{d2}/rgb/0041.png')); ax[1].set_title('ROSE: oracle mask')
for a in ax: a.axis('off')
plt.show()

In [ ]:
# Save edits to Drive; refit + scoring happen in the Phase-4 notebook
import shutil, os
OUT = f'{DRIVE}/checkpoints/phase4_rose'
os.makedirs(OUT, exist_ok=True)
for tag in ('rose_objectmask', 'rose_oraclemask'):
    shutil.copytree(f'/content/edited_{tag}', f'{OUT}/edited_{tag}', dirs_exist_ok=True)
print('saved to', OUT)